# 📘 04_lakehouse_monitoring_functions.ipynb

In this notebook, we create **custom SQL functions (UDFs)** that act as reusable building blocks for Databricks Lakehouse Monitoring.

Databricks Lakehouse Monitoring allows users to define **custom metrics** through SQL expressions or functions.  
While simple metrics like averages or null ratios can be expressed inline, enterprise-scale monitoring often involves repeating the same logic across hundreds of tables — such as:

- Ensuring `end_date >= start_date`
- Checking that `claim_status = 'CLOSED'` implies `closed_at IS NOT NULL`
- Detecting negative or out-of-range numeric values
- Verifying timestamps are not in the future

If each metric embeds this logic directly, maintenance quickly becomes painful and inconsistent.  
By defining SQL functions once and referencing them across metrics, you achieve **reusability**, **consistency**, and **auditability** — all governed under Unity Catalog.

### Why Use Functions

| Benefit | Description |
|----------|-------------|
| **Reusability** | Define logic once and reuse across hundreds of metrics. |
| **Consistency** | Every monitor references the same version of the rule logic. |
| **Auditability** | Functions are Unity Catalog–governed with lineage and versioning. |
| **Modularity** | Update logic centrally without touching metric definitions. |

Example — instead of embedding logic inline:
avg(dbdemos_steventan.monitoring_admin.rule_negative_amount_ratio_bit(amount))

we can reference the function directly in metric templates.

### What We’ll Implement

We’ll define a set of reusable SQL functions grouped by **data quality dimensions**, similar to frameworks like Great Expectations or Soda, but implemented natively in the Lakehouse.

| Dimension | Description | Example Function |
|------------|--------------|------------------|
| **Validity** | Ensure values follow business or logical rules | `rule_negative_amount_ratio_bit(amount)` |
| **Completeness** | Detect missing or empty data | `rule_missing_value_ratio_bit(val)` |
| **Consistency** | Validate relationships between columns | `rule_inconsistent_closed_claims_ratio_bit(status, closed_at)` |
| **Accuracy** | Flag values outside expected business ranges | `rule_premium_out_of_range_ratio_bit(amount)` |

Each rule has two variants:
- `*_ratio_bit` → returns `1.0` or `0.0` for violation ratio aggregation  
- `*_details` → returns a structured JSON object describing violations

### How It Fits in the Metadata Framework

These functions form the **foundation** of the metadata-driven Lakehouse Monitoring framework:

<pre>
┌────────────────────┐
│  SQL Functions     │  ← reusable rule logic (this notebook)
├────────────────────┤
│  Metric Templates  │  ← define metric expressions using these functions
├────────────────────┤
│  Metric Bindings   │  ← attach templates to specific tables
├────────────────────┤
│  Monitors Control  │  ← schedule, enable, and manage monitors
└────────────────────┘
</pre>

## 1️⃣ Widgets — Catalog and Schema Parameters

Define your working context (catalog, schemas, and asset directories).  
These widgets ensure this notebook can run across different workspaces without modification.

In [0]:
dbutils.widgets.text("catalog", "dbdemos_steventan", "Catalog")
dbutils.widgets.text("admin_schema", "monitoring_admin", "Admin Schema")

catalog = dbutils.widgets.get("catalog")
admin_schema = dbutils.widgets.get("admin_schema")

In [0]:
# Uses existing widgets/vars: catalog, admin_schema
FUNC_SCHEMA = f"{catalog}.{admin_schema}"

# Ensure schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FUNC_SCHEMA}")

# Helper to fully-qualify function names
def F(name: str) -> str:
    return f"{FUNC_SCHEMA}.{name}"

## Validity rules

These catch values or combinations that are **invalid** with respect to business logic or formats:
- `end_before_start_*`: end date earlier than start date
- `future_timestamp_*`: timestamp lies in the future
- `negative_amount_*`: negative numeric amounts
- `invalid_date_string_*`: string cannot be parsed to a date (format aware)
- `unexpected_category_*`: value not in an expected set

In [0]:
stmts_validity = [

# 1) end_before_start
f"""CREATE OR REPLACE FUNCTION {F("rule_end_before_start_ratio_bit")}(
  startd DATE, endd DATE
) RETURNS DOUBLE
RETURN CASE WHEN endd < startd THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_end_before_start_details")}(
  pk STRING, startd DATE, endd DATE
) RETURNS STRUCT<pk:STRING,start_date:DATE,end_date:DATE,reason:STRING>
RETURN CASE WHEN endd < startd
            THEN named_struct('pk', pk, 'start_date', startd, 'end_date', endd, 'reason', 'end_before_start')
       END""",

# 2) future_timestamp
f"""CREATE OR REPLACE FUNCTION {F("rule_future_timestamp_ratio_bit")}(
  ts TIMESTAMP
) RETURNS DOUBLE
RETURN CASE WHEN ts > current_timestamp() THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_future_timestamp_details")}(
  pk STRING, ts TIMESTAMP
) RETURNS STRUCT<pk:STRING,ts:TIMESTAMP,reason:STRING>
RETURN CASE WHEN ts > current_timestamp()
            THEN named_struct('pk', pk, 'ts', ts, 'reason', 'future_timestamp')
       END""",

# 3) negative_amount
f"""CREATE OR REPLACE FUNCTION {F("rule_negative_amount_ratio_bit")}(
  amount DOUBLE
) RETURNS DOUBLE
RETURN CASE WHEN amount < 0 THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_negative_amount_details")}(
  pk STRING, amount DOUBLE
) RETURNS STRUCT<pk:STRING,amount:DOUBLE,reason:STRING>
RETURN CASE WHEN amount < 0
            THEN named_struct('pk', pk, 'amount', amount, 'reason', 'negative_amount')
       END""",

# 4) invalid_date_string (malformed per fmt)
f"""CREATE OR REPLACE FUNCTION {F("rule_invalid_date_string_ratio_bit")}(
  date_str STRING, fmt STRING
) RETURNS DOUBLE
RETURN CASE WHEN to_date(date_str, fmt) IS NULL THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_invalid_date_string_details")}(
  pk STRING, date_str STRING, fmt STRING
) RETURNS STRUCT<pk:STRING,date_str:STRING,fmt:STRING,reason:STRING>
RETURN CASE WHEN to_date(date_str, fmt) IS NULL
            THEN named_struct('pk', pk, 'date_str', date_str, 'fmt', fmt, 'reason', 'invalid_date_string')
       END""",

# 5) unexpected_category
f"""CREATE OR REPLACE FUNCTION {F("rule_unexpected_category_ratio_bit")}(
  val STRING, csv_expected STRING
) RETURNS DOUBLE
RETURN CASE WHEN NOT array_contains(split(csv_expected, ','), val) THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_unexpected_category_details")}(
  pk STRING, val STRING, csv_expected STRING
) RETURNS STRUCT<pk:STRING,val:STRING,expected_set:STRING,reason:STRING>
RETURN CASE WHEN NOT array_contains(split(csv_expected, ','), val)
            THEN named_struct('pk', pk, 'val', val, 'expected_set', csv_expected, 'reason', 'unexpected_category')
       END""",
]

## Completeness rules

These detect **missingness** (null/empty) in mandatory fields.

In [0]:
stmts_completeness = [

# 6) missing_value (NULL or empty string)
f"""CREATE OR REPLACE FUNCTION {F("rule_missing_value_ratio_bit")}(
  val STRING
) RETURNS DOUBLE
RETURN CASE WHEN val IS NULL OR trim(val) = '' THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_missing_value_details")}(
  pk STRING, colname STRING, val STRING
) RETURNS STRUCT<pk:STRING,colname:STRING,reason:STRING>
RETURN CASE WHEN val IS NULL OR trim(val) = ''
            THEN named_struct('pk', pk, 'colname', colname, 'reason', 'missing_value')
       END""",
]

## Consistency rules

These enforce **logical consistency** across related fields.

In [0]:
stmts_consistency = [

# 7) closed claims must have closed_at
f"""CREATE OR REPLACE FUNCTION {F("rule_inconsistent_closed_claims_ratio_bit")}(
  claim_status STRING, closed_at TIMESTAMP
) RETURNS DOUBLE
RETURN CASE WHEN claim_status = 'CLOSED' AND closed_at IS NULL THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_inconsistent_closed_claims_details")}(
  pk STRING, claim_status STRING, closed_at TIMESTAMP
) RETURNS STRUCT<pk:STRING,claim_status:STRING,closed_at:TIMESTAMP,reason:STRING>
RETURN CASE WHEN claim_status = 'CLOSED' AND closed_at IS NULL
            THEN named_struct('pk', pk, 'claim_status', claim_status, 'closed_at', closed_at, 'reason', 'inconsistent_closed_claims')
       END""",
]

## Accuracy rules

These flag values that are **outside expected business ranges**.

In [0]:
stmts_accuracy = [

# 8) premium amount out of expected business range (example thresholds)
f"""CREATE OR REPLACE FUNCTION {F("rule_premium_out_of_range_ratio_bit")}(
  amount DOUBLE
) RETURNS DOUBLE
RETURN CASE WHEN amount < 100 OR amount > 100000 THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_premium_out_of_range_details")}(
  pk STRING, amount DOUBLE
) RETURNS STRUCT<pk:STRING,amount:DOUBLE,reason:STRING>
RETURN CASE WHEN amount < 100 OR amount > 100000
            THEN named_struct('pk', pk, 'amount', amount, 'reason', 'premium_out_of_range')
       END""",
]

## Execute — Create/Replace all functions

Re-running this cell is **idempotent** (safe). It recreates the UDFs in `catalog.admin_schema`.

In [0]:
for s in (stmts_validity + stmts_completeness + stmts_consistency + stmts_accuracy):
    spark.sql(s)

print(f"✅ Created/updated custom metric functions in {FUNC_SCHEMA}")

### Next Step

After running this notebook:
1. You’ll have a library of SQL functions under `{catalog}.{admin_schema}`.
2. These will be referenced by **metric templates** in the next notebook (v0.2).
3. This approach enables modular, metadata-driven monitoring at scale — with clear separation of **rule logic**, **metric definition**, and **monitor orchestration**.